In [ ]:
import os
import json

from glob import glob

In [ ]:
COCO_CATEGORIES = [
    "person", "bicycle", "car", "motorcycle", "airplane", "bus", "train", "truck", "boat", "traffic light", "fire hydrant", "stop sign", "parking meter", "bench", "bird", 
    "cat", "dog", "horse", "sheep", "cow", "elephant", "bear", "zebra", "giraffe", "backpack", "umbrella", "handbag", "tie", "suitcase", "frisbee", "skis", "snowboard", 
    "sports ball", "kite", "baseball bat", "baseball glove", "skateboard", "surfboard", "tennis racket", "bottle", "wine glass", "cup", "fork", "knife", "spoon", "bowl", 
    "banana", "apple", "sandwich", "orange", "broccoli", "carrot", "hot dog", "pizza", "donut", "cake", "chair", "couch", "potted plant", "bed", "dining table", "toilet", 
    "tv", "laptop", "mouse", "remote", "keyboard", "cell phone", "microwave", "oven", "toaster", "sink", "refrigerator", "book", "clock", "vase", "scissors", "teddy bear", 
    "hair drier", "toothbrush"
]

print(f"Total number of COCO categories: {len(COCO_CATEGORIES)}")

In [ ]:
json_files = glob(os.path.join("val", "*.json"))

In [ ]:
for json_file in json_files:
    with open(json_file) as f:
        data = json.load(f)

    per_class_instance_count = {}
    clsId_to_clsName = {cat["id"]: cat["name"] for cat in data["categories"]}

    for annotation in data["annotations"]:
        clsId = annotation["category_id"]
        cls_name = clsId_to_clsName[clsId]
        if cls_name.lower() in COCO_CATEGORIES:
            print(f"Info: Class '{cls_name}' from COCO categories found in {json_file}.")
            per_class_instance_count[cls_name] = per_class_instance_count.get(cls_name, 0) + 1

    print(f"File: {json_file}")
    print(f"Total number of classes: {len(per_class_instance_count)}")
    print("Per-class instance count:")
    for cls_name, count in per_class_instance_count.items():
        print(f"{cls_name}: {count}")
    print("\n")

    

In [ ]:
from __future__ import annotations

import csv
import json
import os
import random
from collections import defaultdict
from glob import glob
from pathlib import Path

In [ ]:
ANNOTATION_GLOB = "val/*.json"

IMAGES_PER_CLASS = 100
RANDOM_SEED = 42

OUTPUT_DIR = Path("val/coco_aligned_subsets")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

rng = random.Random(RANDOM_SEED)


# Official MS COCO 2017 detection category IDs.
# The IDs are intentionally non-contiguous.
COCO_CATEGORY_ID_BY_NAME = {
    "person": 1,
    "bicycle": 2,
    "car": 3,
    "motorcycle": 4,
    "airplane": 5,
    "bus": 6,
    "train": 7,
    "truck": 8,
    "boat": 9,
    "traffic light": 10,
    "fire hydrant": 11,
    "stop sign": 13,
    "parking meter": 14,
    "bench": 15,
    "bird": 16,
    "cat": 17,
    "dog": 18,
    "horse": 19,
    "sheep": 20,
    "cow": 21,
    "elephant": 22,
    "bear": 23,
    "zebra": 24,
    "giraffe": 25,
    "backpack": 27,
    "umbrella": 28,
    "handbag": 31,
    "tie": 32,
    "suitcase": 33,
    "frisbee": 34,
    "skis": 35,
    "snowboard": 36,
    "sports ball": 37,
    "kite": 38,
    "baseball bat": 39,
    "baseball glove": 40,
    "skateboard": 41,
    "surfboard": 42,
    "tennis racket": 43,
    "bottle": 44,
    "wine glass": 46,
    "cup": 47,
    "fork": 48,
    "knife": 49,
    "spoon": 50,
    "bowl": 51,
    "banana": 52,
    "apple": 53,
    "sandwich": 54,
    "orange": 55,
    "broccoli": 56,
    "carrot": 57,
    "hot dog": 58,
    "pizza": 59,
    "donut": 60,
    "cake": 61,
    "chair": 62,
    "couch": 63,
    "potted plant": 64,
    "bed": 65,
    "dining table": 67,
    "toilet": 70,
    "tv": 72,
    "laptop": 73,
    "mouse": 74,
    "remote": 75,
    "keyboard": 76,
    "cell phone": 77,
    "microwave": 78,
    "oven": 79,
    "toaster": 80,
    "sink": 81,
    "refrigerator": 82,
    "book": 84,
    "clock": 85,
    "vase": 86,
    "scissors": 87,
    "teddy bear": 88,
    "hair drier": 89,
    "toothbrush": 90,
}

COCO_CATEGORY_NAME_BY_ID = {
    category_id: category_name
    for category_name, category_id in COCO_CATEGORY_ID_BY_NAME.items()
}

COCO_CATEGORY_IDS = set(COCO_CATEGORY_ID_BY_NAME.values())

COCO_CATEGORIES = [
    {
        "id": category_id,
        "name": category_name,
        "supercategory": "object",
    }
    for category_name, category_id in COCO_CATEGORY_ID_BY_NAME.items()
]

# Objects365 label variants mapped to official MS COCO labels.
#
# Exact matches use their normalized Objects365 name automatically.
# Add a mapping here only after verifying that the Objects365 category has
# the same semantic definition as the corresponding COCO category.
OBJECTS365_TO_COCO_NAME = {
    "person": "person",
    "bicycle": "bicycle",
    "car": "car",
    "motorcycle": "motorcycle",
    "airplane": "airplane",
    "bus": "bus",
    "train": "train",
    "truck": "truck",
    "boat": "boat",
    "traffic light": "traffic light",
    "fire hydrant": "fire hydrant",
    "stop sign": "stop sign",
    "parking meter": "parking meter",
    "bench": "bench",
    "bird": "bird",
    "cat": "cat",
    "dog": "dog",
    "horse": "horse",
    "sheep": "sheep",
    "cow": "cow",
    "elephant": "elephant",
    "bear": "bear",
    "zebra": "zebra",
    "giraffe": "giraffe",
    "backpack": "backpack",
    "umbrella": "umbrella",
    "handbag": "handbag",
    "tie": "tie",
    "suitcase": "suitcase",
    "frisbee": "frisbee",
    "skis": "skis",
    "snowboard": "snowboard",
    "sports ball": "sports ball",
    "kite": "kite",
    "baseball bat": "baseball bat",
    "baseball glove": "baseball glove",
    "skateboard": "skateboard",
    "surfboard": "surfboard",
    "tennis racket": "tennis racket",
    "bottle": "bottle",
    "wine glass": "wine glass",
    "cup": "cup",
    "fork": "fork",
    "knife": "knife",
    "spoon": "spoon",
    "bowl": "bowl",
    "banana": "banana",
    "apple": "apple",
    "sandwich": "sandwich",
    "orange": "orange",
    "broccoli": "broccoli",
    "carrot": "carrot",
    "hot dog": "hot dog",
    "pizza": "pizza",
    "donut": "donut",
    "cake": "cake",
    "chair": "chair",
    "sofa": "couch",
    "couch": "couch",
    "potted plant": "potted plant",
    "pottedplant": "potted plant",
    "bed": "bed",
    "dining table": "dining table",
    "toilet": "toilet",
    "tv": "tv",
    "television": "tv",
    "laptop": "laptop",
    "mouse": "mouse",
    "remote": "remote",
    "keyboard": "keyboard",
    "cell phone": "cell phone",
    "cellphone": "cell phone",
    "mobile phone": "cell phone",
    "microwave": "microwave",
    "oven": "oven",
    "toaster": "toaster",
    "sink": "sink",
    "refrigerator": "refrigerator",
    "book": "book",
    "clock": "clock",
    "vase": "vase",
    "scissors": "scissors",
    "teddy bear": "teddy bear",
    "hair drier": "hair drier",
    "hair dryer": "hair drier",
    "toothbrush": "toothbrush",
}

In [ ]:
def normalise_class_name(name: str) -> str:
    return " ".join(name.strip().lower().split())

In [ ]:
def build_objects365_to_coco_mapping(categories: list[dict]) -> tuple[
    dict[int, int],
    dict[int, str],
    list[dict],
]:
    """
    Build:
    - Objects365 source category ID -> official COCO category ID
    - Objects365 source category ID -> official COCO name
    - Full mapping report for all source categories
    """
    objects365_id_to_coco_id = {}
    objects365_id_to_coco_name = {}
    mapping_report = []

    for category in sorted(categories, key=lambda item: item["id"]):
        objects365_category_id = category["id"]
        objects365_category_name = category["name"]

        normalised_name = normalise_class_name(objects365_category_name)

        coco_category_name = OBJECTS365_TO_COCO_NAME.get(
            normalised_name,
            normalised_name,
        )

        if coco_category_name in COCO_CATEGORY_ID_BY_NAME:
            coco_category_id = COCO_CATEGORY_ID_BY_NAME[coco_category_name]

            objects365_id_to_coco_id[objects365_category_id] = coco_category_id
            objects365_id_to_coco_name[objects365_category_id] = coco_category_name

            mapping_report.append(
                {
                    "objects365_category_id": objects365_category_id,
                    "objects365_category_name": objects365_category_name,
                    "coco_category_id": coco_category_id,
                    "coco_category_name": coco_category_name,
                    "mapping_status": "included",
                }
            )
        else:
            mapping_report.append(
                {
                    "objects365_category_id": objects365_category_id,
                    "objects365_category_name": objects365_category_name,
                    "coco_category_id": "",
                    "coco_category_name": "",
                    "mapping_status": "excluded",
                }
            )

    return (
        objects365_id_to_coco_id,
        objects365_id_to_coco_name,
        mapping_report,
    )

In [ ]:
def write_mapping_report_csv(
    mapping_report: list[dict],
    output_path: Path,
) -> None:
    fields = [
        "objects365_category_id",
        "objects365_category_name",
        "coco_category_id",
        "coco_category_name",
        "mapping_status",
    ]

    with output_path.open("w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fields)
        writer.writeheader()
        writer.writerows(mapping_report)

In [ ]:
def write_selection_report_csv(
    rows: list[dict],
    output_path: Path,
) -> None:
    fields = [
        "objects365_category_id",
        "objects365_category_name",
        "coco_category_id",
        "coco_category_name",
        "available_images",
        "selected_images_containing_class",
        "target_images",
        "target_reached",
    ]

    with output_path.open("w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fields)
        writer.writeheader()
        writer.writerows(rows)


In [ ]:
def validate_coco_subset(subset_data: dict) -> None:
    """Validate category references and official COCO-ID remapping."""
    output_category_ids = {
        category["id"]
        for category in subset_data["categories"]
    }

    annotation_category_ids = {
        annotation["category_id"]
        for annotation in subset_data["annotations"]
    }

    annotation_image_ids = {
        annotation["image_id"]
        for annotation in subset_data["annotations"]
    }

    output_image_ids = {
        image["id"]
        for image in subset_data["images"]
    }

    unknown_annotation_category_ids = (
        annotation_category_ids - output_category_ids
    )

    non_coco_category_ids = (
        annotation_category_ids - COCO_CATEGORY_IDS
    )

    unknown_annotation_image_ids = (
        annotation_image_ids - output_image_ids
    )

    assert not unknown_annotation_category_ids, (
        "Some annotations refer to category IDs absent from "
        "the output categories list: "
        f"{sorted(unknown_annotation_category_ids)}"
    )

    assert not non_coco_category_ids, (
        "Some annotations still use non-COCO category IDs: "
        f"{sorted(non_coco_category_ids)}"
    )

    assert not unknown_annotation_image_ids, (
        "Some annotations refer to image IDs absent from "
        "the output images list: "
        f"{sorted(unknown_annotation_image_ids)[:20]}"
    )

In [ ]:
json_files = sorted(glob(ANNOTATION_GLOB))

if not json_files:
    raise FileNotFoundError(
        f"No JSON files found with pattern: {ANNOTATION_GLOB}"
    )

print(f"Found {len(json_files)} annotation JSON file(s).")
print(f"Images per class target: {IMAGES_PER_CLASS}")
print(f"Random seed: {RANDOM_SEED}")
print(f"Output directory: {OUTPUT_DIR.resolve()}")


# %%
for json_file in json_files:
    print("\n" + "=" * 90)
    print(f"Processing: {json_file}")
    print("=" * 90)

    with open(json_file, "r", encoding="utf-8") as f:
        data = json.load(f)

    source_categories = data.get("categories", [])
    source_images = data.get("images", [])
    source_annotations = data.get("annotations", [])

    if not source_categories or not source_images or not source_annotations:
        raise ValueError(
            f"{json_file} does not contain valid COCO-format "
            "categories, images, and annotations."
        )

    clsId_to_clsName = {
        category["id"]: category["name"]
        for category in source_categories
    }

    (
        objects365_id_to_coco_id,
        objects365_id_to_coco_name,
        mapping_report,
    ) = build_objects365_to_coco_mapping(source_categories)

    selected_objects365_category_ids = set(
        objects365_id_to_coco_id.keys()
    )

    selected_coco_category_ids = sorted(
        set(objects365_id_to_coco_id.values())
    )

    print(f"Source categories: {len(source_categories)}")
    print(
        "Mapped Objects365 source categories: "
        f"{len(selected_objects365_category_ids)}"
    )
    print(
        "Mapped official COCO categories: "
        f"{len(selected_coco_category_ids)}"
    )

    if len(selected_coco_category_ids) < 50:
        raise ValueError(
            "Fewer than 50 Objects365 categories were mapped to COCO. "
            "Inspect the mapping dictionary and source category names."
        )

    annotations_by_image = defaultdict(list)
    image_ids_by_objects365_category = defaultdict(set)

    for annotation in source_annotations:
        objects365_category_id = annotation["category_id"]

        if objects365_category_id not in selected_objects365_category_ids:
            continue

        image_id = annotation["image_id"]

        annotations_by_image[image_id].append(annotation)
        image_ids_by_objects365_category[
            objects365_category_id
        ].add(image_id)

    print(
        f"\nSelecting up to {IMAGES_PER_CLASS} unique images "
        "per COCO-overlapping class."
    )

    selected_image_ids = set()
    selection_report = []

    # Recreate an RNG for each source JSON so each JSON is reproducible
    # independently, even if more than one JSON file is processed.
    file_rng = random.Random(RANDOM_SEED)

    for objects365_category_id in sorted(
        selected_objects365_category_ids
    ):
        objects365_category_name = clsId_to_clsName[
            objects365_category_id
        ]

        coco_category_id = objects365_id_to_coco_id[
            objects365_category_id
        ]

        coco_category_name = objects365_id_to_coco_name[
            objects365_category_id
        ]

        candidate_image_ids = sorted(
            image_ids_by_objects365_category[
                objects365_category_id
            ]
        )

        if not candidate_image_ids:
            print(
                f"Warning: no candidate images found for "
                f"Objects365 '{objects365_category_name}' "
                f"mapped to COCO '{coco_category_name}'."
            )
            continue

        current_count = len(
            selected_image_ids.intersection(candidate_image_ids)
        )

        remaining_needed = max(
            0,
            IMAGES_PER_CLASS - current_count,
        )

        available_unselected_ids = [
            image_id
            for image_id in candidate_image_ids
            if image_id not in selected_image_ids
        ]

        file_rng.shuffle(available_unselected_ids)

        selected_image_ids.update(
            available_unselected_ids[:remaining_needed]
        )

        final_selected_count = len(
            selected_image_ids.intersection(candidate_image_ids)
        )

        selection_report.append(
            {
                "objects365_category_id": objects365_category_id,
                "objects365_category_name": objects365_category_name,
                "coco_category_id": coco_category_id,
                "coco_category_name": coco_category_name,
                "available_images": len(candidate_image_ids),
                "selected_images_containing_class": final_selected_count,
                "target_images": IMAGES_PER_CLASS,
                "target_reached": final_selected_count >= IMAGES_PER_CLASS,
            }
        )

    images_by_id = {
        image["id"]: image
        for image in source_images
    }

    missing_source_image_records = sorted(
        selected_image_ids - set(images_by_id)
    )

    if missing_source_image_records:
        raise ValueError(
            "The selected image IDs are missing from the source images list: "
            f"{missing_source_image_records[:20]}"
        )

    selected_images = [
        images_by_id[image_id]
        for image_id in sorted(selected_image_ids)
    ]

    selected_annotations = []

    for annotation in source_annotations:
        image_id = annotation["image_id"]
        objects365_category_id = annotation["category_id"]

        if image_id not in selected_image_ids:
            continue

        if objects365_category_id not in objects365_id_to_coco_id:
            continue

        remapped_annotation = dict(annotation)
        remapped_annotation["category_id"] = (
            objects365_id_to_coco_id[objects365_category_id]
        )

        selected_annotations.append(remapped_annotation)

    output_coco_category_ids = sorted(
        {
            annotation["category_id"]
            for annotation in selected_annotations
        }
    )

    selected_categories = [
        {
            "id": coco_category_id,
            "name": COCO_CATEGORY_NAME_BY_ID[coco_category_id],
            "supercategory": "object",
        }
        for coco_category_id in output_coco_category_ids
    ]

    objects365_to_coco_mapping_metadata = {
        str(objects365_category_id): {
            "objects365_name": clsId_to_clsName[
                objects365_category_id
            ],
            "coco_id": coco_category_id,
            "coco_name": COCO_CATEGORY_NAME_BY_ID[coco_category_id],
        }
        for objects365_category_id, coco_category_id in sorted(
            objects365_id_to_coco_id.items()
        )
    }

    subset_data = {
        "info": {
            **data.get("info", {}),
            "description": (
                "Objects365 validation subset filtered to COCO-overlapping "
                "classes and remapped to official MS COCO category IDs. "
                f"The selection target was {IMAGES_PER_CLASS} images per "
                "source category."
            ),
            "source_annotation_file": os.path.basename(json_file),
            "selection_seed": RANDOM_SEED,
            "images_per_class_target": IMAGES_PER_CLASS,
            "objects365_to_coco_category_id_mapping": (
                objects365_to_coco_mapping_metadata
            ),
        },
        "licenses": data.get("licenses", []),
        "images": selected_images,
        "annotations": selected_annotations,
        "categories": selected_categories,
    }

    validate_coco_subset(subset_data)

    source_stem = Path(json_file).stem

    output_json_path = OUTPUT_DIR / (
        f"{source_stem}_coco_remapped_"
        f"{IMAGES_PER_CLASS}_images_per_class.json"
    )

    mapping_report_path = OUTPUT_DIR / (
        f"{source_stem}_objects365_to_coco_mapping.csv"
    )

    selection_report_path = OUTPUT_DIR / (
        f"{source_stem}_selection_report_"
        f"{IMAGES_PER_CLASS}_images_per_class.csv"
    )

    class_list_path = OUTPUT_DIR / (
        f"{source_stem}_selected_coco_classes.txt"
    )

    image_ids_path = OUTPUT_DIR / (
        f"{source_stem}_selected_image_ids.txt"
    )

    with output_json_path.open("w", encoding="utf-8") as f:
        json.dump(subset_data, f, ensure_ascii=False, indent=2)

    write_mapping_report_csv(
        mapping_report=mapping_report,
        output_path=mapping_report_path,
    )

    write_selection_report_csv(
        rows=selection_report,
        output_path=selection_report_path,
    )

    with class_list_path.open("w", encoding="utf-8") as f:
        for coco_category_id in output_coco_category_ids:
            f.write(
                f"{coco_category_id}\t"
                f"{COCO_CATEGORY_NAME_BY_ID[coco_category_id]}\n"
            )

    with image_ids_path.open("w", encoding="utf-8") as f:
        for image_id in sorted(selected_image_ids):
            f.write(f"{image_id}\n")

    classes_reaching_target = [
        row
        for row in selection_report
        if row["target_reached"]
    ]

    classes_below_target = [
        row
        for row in selection_report
        if not row["target_reached"]
    ]

    print("\nSubset creation completed.")
    print(f"Unique selected images: {len(selected_images)}")
    print(f"Selected remapped annotations: {len(selected_annotations)}")
    print(f"Output COCO categories: {len(selected_categories)}")
    print(
        f"Classes reaching {IMAGES_PER_CLASS} images: "
        f"{len(classes_reaching_target)}"
    )
    print(
        f"Classes below {IMAGES_PER_CLASS} images: "
        f"{len(classes_below_target)}"
    )
    print(f"COCO-format subset JSON: {output_json_path}")
    print(f"Objects365 to COCO mapping report: {mapping_report_path}")
    print(f"Per-class selection report: {selection_report_path}")
    print(f"Selected COCO class list: {class_list_path}")
    print(f"Selected image IDs: {image_ids_path}")

    print("\nPer-class selection summary:")
    for row in selection_report:
        status = "OK" if row["target_reached"] else "INSUFFICIENT"

        print(
            f"[{status}] "
            f"Objects365 ID {row['objects365_category_id']} "
            f"'{row['objects365_category_name']}' "
            f"-> COCO ID {row['coco_category_id']} "
            f"'{row['coco_category_name']}': "
            f"{row['selected_images_containing_class']}/"
            f"{row['target_images']} selected, "
            f"{row['available_images']} available"
        )

    if classes_below_target:
        print("\nWarning: some mapped categories have fewer than "
              f"{IMAGES_PER_CLASS} selected images.")
        print(
            "For a strictly balanced final evaluation subset, exclude these "
            "categories or lower the fixed per-class threshold for all "
            "classes before any model comparison."
        )